*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 3: Autograd & the Computational Graph. To understand how gradients are built, tracked, and detached during training, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

Autograd is the engine behind backpropagation in PyTorch. Each tensor operation records a small piece of the computational graph, and the backward pass follows that graph to compute gradients automatically.

## 1. The Dynamic Computation Graph
### Step 1: Inspect Leaves and Computation History

In [ ]:
import torch

# A computation graph is a DAG of tensor operations recorded by Autograd.
# Leaves are tensors created directly (grad_fn=None).
# Operations create intermediate tensors with grad_fn pointing to the backward function.
# Each forward pass builds a new graph for the operations that actually execute.
W = torch.randn(3, 3, requires_grad=True)
x = torch.randn(3, 1)  # requires_grad=False. Still a leaf.

# 2. Forward pass: matrix multiplication adds a graph node
z = torch.matmul(W, x)
print(f"z has grad_fn: {z.grad_fn}")  # <MmBackward0 object>

# 3. A reduction adds another node
loss = z.sum()
print(f"loss has grad_fn: {loss.grad_fn}")  # <SumBackward0 object>

assert W.is_leaf and W.grad_fn is None
assert x.is_leaf and x.grad_fn is None
assert not z.is_leaf and z.grad_fn is not None
assert not loss.is_leaf and loss.grad_fn is not None

z has grad_fn: <MmBackward0 object at 0x000001B7B28D3A30>
loss has grad_fn: <SumBackward0 object at 0x000001B7B2750850>


### Step 2: Observe Define-by-Run Behavior

In [ ]:
import torch

# Define-by-Run: the graph is built as code executes, not ahead of time.
# Python control flow determines which operations are recorded.
# Different inputs can follow different branches and record different backward paths.
def adaptive_transform(x):
    if x.mean().item() >= 0:  # This decision uses .item() and lies outside the graph
        return x.square()
    return x.abs()

positive_input = torch.tensor(
    [1.0, 2.0], requires_grad=True
)
negative_input = torch.tensor(
    [-1.0, -2.0], requires_grad=True
)

positive_output = adaptive_transform(positive_input)
negative_output = adaptive_transform(negative_input)

print(positive_output.grad_fn)
print(negative_output.grad_fn)

## 2. The Engine: `backward()` and `requires_grad`
### Step 1: Seed Scalar and Vector Backward Passes

In [ ]:
import torch

# backward() computes a Vector-Jacobian Product (VJP) to avoid constructing the full Jacobian.
# A scalar output uses an implicit incoming gradient of 1.
# A vector output requires an explicit gradient tensor or a reduction to scalar.
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])
loss = (w * x - 7.0).pow(2).sum()
loss.backward()
assert torch.equal(w.grad, torch.tensor([-6.0]))

# Vector output: provide the incoming gradient explicitly.
u = torch.tensor([1.0, 2.0], requires_grad=True)
v = u.square()
incoming_gradient = torch.tensor([1.0, 0.5])
v.backward(incoming_gradient)
assert torch.equal(u.grad, torch.tensor([2.0, 2.0]))

### Step 2: Observe Gradient Accumulation

In [ ]:
import torch

# Gradients accumulate in .grad buffers; each backward() adds to existing values.
# This enables deliberate accumulation but also requires explicit zero_grad() between optimization steps.
# Assigning None to w.grad discards the buffer; using None is often faster than filling with zeros.
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])
observed_gradients = []

for iteration in range(3):
    y = w * x
    y.backward()
    print(f"Iteration {iteration + 1} - w.grad: {w.grad}")
    observed_gradients.append(w.grad.item())

# tensor([3.]), then tensor([6.]), then tensor([9.])
assert observed_gradients == [3.0, 6.0, 9.0]

# Reset the buffer before an unrelated optimization step.
w.grad = None
assert w.grad is None

Iteration 1 - w.grad: tensor([3.])
Iteration 2 - w.grad: tensor([6.])
Iteration 3 - w.grad: tensor([9.])


## 3. Breaking the Graph: Memory & Performance Management
### Step 1: Compare Gradient-Tracking Contexts

In [ ]:
import torch

# no_grad() disables graph recording for efficiency during inference or validation.
# inference_mode() provides additional optimizations and removes bookkeeping overhead.
# Both still execute the operations normally; they just don't record them for differentiation.
source = torch.tensor([1.0, 2.0], requires_grad=True)
tracked = source.square()

with torch.no_grad():
    no_grad_result = source.square()

with torch.inference_mode():
    inference_result = source.square()


assert tracked.requires_grad and tracked.grad_fn is not None
assert not inference_result.requires_grad and inference_result.grad_fn is None
assert not no_grad_result.requires_grad and no_grad_result.grad_fn is None
assert not inference_result.requires_grad and inference_result.grad_fn is None

### Step 2: Detach One Tensor from Its History

In [ ]:
import torch

# detach() disconnects a single tensor from its graph history.
# The detached tensor shares storage with the source but has grad_fn=None and requires_grad=False.
# This is useful for creating a target tensor that won't accumulate gradients.
x = torch.tensor([1.0, 2.0], requires_grad=True)
y_pred = (x * 2).sigmoid()
detached = y_pred.detach()

assert y_pred.grad_fn is not None
assert detached.grad_fn is None
assert not detached.requires_grad
assert detached.data_ptr() == y_pred.data_ptr()

# Move to CPU before exposing data to NumPy.
y_pred_np = detached.cpu().numpy()

### Step 3: Record Metrics Without Retaining Graphs

In [ ]:
import torch

# Using .item() to extract scalars breaks the graph reference.
# This prevents VRAM leaks when accumulating metrics across many batches.
# Tensors accumulate graphs; Python floats do not.
parameter = torch.tensor([2.0], requires_grad=True)
loss_history = []

for target in (5.0, 6.0, 7.0):
    loss = (parameter * 3.0 - target).pow(2).mean()
    loss_history.append(loss.item())

assert all(isinstance(value, float) for value in loss_history)

## 4. Gotchas & Reality Checks: Production Pitfalls
### Gotcha 1: In-Place Operations on Graph Leaves

In [ ]:
import torch

# In-place operations (with _) directly modify existing storage.
# Autograd forbids in-place mutations on gradient-tracked leaves because they break graph consistency.
# The solution is to use out-of-place operations (e.g., a + 1 instead of a.add_(1)).
a = torch.tensor([2.0], requires_grad=True)

try:
    a.add_(1.0)
except RuntimeError as error:
    print(f"Expected leaf mutation error: {error}")

# Create a new non-leaf tensor instead of mutating a.
safe_value = a + 1.0
objective = (safe_value * 3.0).sum()
objective.backward()

assert a.is_leaf
assert torch.equal(a.grad, torch.tensor([3.0]))

Expected leaf mutation error: a leaf Variable that requires grad is being used in an in-place operation.


### Gotcha 2: Conversions, Device Copies, and Leaf Status

In [ ]:
import torch

# to(dtype=...) on a tensor requiring gradients creates a new tensor with grad_fn.
# This means the result is no longer a leaf, even though it was created directly.
# To keep a tensor as a leaf, allocate it with the final dtype and device from the start.
source = torch.rand(3, 3, requires_grad=True)
converted = source.to(dtype=torch.float64)
print(converted.is_leaf)  # False

# Allocate directly with the final configuration.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
direct = torch.rand(
    3, 3,
    dtype=torch.float64,
    device=device,
    requires_grad=True,
)

assert not converted.is_leaf
assert converted.grad_fn is not None
assert direct.is_leaf and direct.grad_fn is None

False
